# Imports

In [ ]:
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
import xgboost as xgb

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, roc_curve

In [ ]:
np.set_printoptions(precision=4, linewidth=120, suppress=True, floatmode='fixed')

# Objective

Predict whether or not an industrial machine requires maintenance based on 178 IoT (Internet of Things) sensor data readings.

# Dataset

Each row of the dataset contains 178 readings, representing columns from different sensors. In other words, there are 178 columns comprising the IoT sensor readings.

In [ ]:
data_path = f"data"

In [ ]:
df = pd.read_csv(f"{data_path}/dataset.csv")

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

# Exploratory Analysis

## Class Distribution

Prevalence is the percentage of samples that possess the characteristic you are trying to predict. In this specific scenario, it means that machines requiring maintenance represent the positive class (occurrence of the event), while those not requiring maintenance represent the negative class (non-occurrence of the event).

In [ ]:
def dist_classes(series):
    positive = sum(series) / len(series)
    s = pd.Series(data = [1 - positive, positive], index=['Class 0', 'Classe 1'])
    return s

In [ ]:
dist_classes(df['LABEL'])

In [ ]:
cols = df.columns[:-1].tolist()

In [ ]:
cols_duplicadas = [x for x in cols if cols.count(x) > 1]
cols_duplicadas

The assert statement in Python is a debugging aid used to check if a given condition is true.\
If the condition evaluates to False, an AssertionError is raised, halting the program's execution. 

In [ ]:
assert len(cols_duplicadas) == 0, 'found duplicated features'

# Split Data

In [ ]:
df_valid_test = df.sample(frac=0.3)

In [ ]:
df_test = df_valid_test.sample(frac = 0.5)

In [ ]:
df_valid = df_valid_test.drop(labels = df_test.index)

In [ ]:
df_train = df.drop(labels = df_valid_test.index)

In [ ]:
df_train.shape, df_valid.shape, df_test.shape

In [ ]:
dist_classes(df_train['LABEL'])

In [ ]:
dist_classes(df_valid['LABEL'])

In [ ]:
dist_classes(df_test['LABEL'])

# Class Balancing

REMINDER: Perform class balancing only on the training data!

Method: undersampling

In [ ]:
cls_positive = df_train['LABEL'] == 1

In [ ]:
df_treino_pos = df_train[cls_positive]

In [ ]:
df_treino_pos.shape

In [ ]:
df_treino_neg = df_train[~cls_positive]

In [ ]:
df_treino_neg.shape

In [ ]:
n = min(len(df_treino_pos), len(df_treino_neg))
n

In [ ]:
df_train_final = pd.concat([df_treino_pos.sample(n = n, replace=False, random_state=42),
                            df_treino_neg.sample(n = n, replace=False, random_state=42)],
                            axis = 0,
                            ignore_index=True)

In [ ]:
df_train_final.info()

In [ ]:
dist_classes(df_train_final['LABEL'])

In [ ]:
X_train = df_train_final[cols]
y_train = df_train_final['LABEL']

In [ ]:
X_valid = df_valid[cols]
y_valid = df_valid['LABEL']

In [ ]:
X_test = df_test[cols]
y_test = df_test['LABEL']

# Scaling

In [ ]:
scaler = StandardScaler()

In [ ]:
scaler.fit(X_train)

In [ ]:
X_train_tf = scaler.transform(X_train)
X_valid_tf = scaler.transform(X_valid)
X_test_tf = scaler.transform(X_test)

# Metrics

In [ ]:
def print_report(y_actual, y_pred, thresh):
    
    auc = roc_auc_score(y_actual, y_pred)
    
    accuracy = accuracy_score(y_actual, (y_pred > thresh))
    
    recall = recall_score(y_actual, (y_pred > thresh))
    
    precision = precision_score(y_actual, (y_pred > thresh))
    
    print('AUC:%.3f'%auc)
    print('Accuracy:%.3f'%accuracy)
    print('Recall:%.3f'%recall)
    print('Precision:%.3f'%precision)
    print(' ')
    
    return None

# Predictive Modeling

## Model 1: Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=500, random_state=142)

In [ ]:
lr.fit(X_train_tf, y_train)

In [ ]:
y_train_pred_lr = lr.predict(X_train_tf)
y_valid_pred_lr = lr.predict(X_valid_tf)

In [ ]:
print_report(y_train, y_train_pred_lr, thresh=0.5)

In [ ]:
print_report(y_valid, y_valid_pred_lr, thresh=0.5)

## Model 2: XGBoost

In [ ]:
xgboost = XGBClassifier()

In [ ]:
xgboost.fit(X_train_tf, y_train)

In [ ]:
y_train_pred_xg = xgboost.predict(X_train_tf)
y_valid_pred_xg = xgboost.predict(X_valid_tf)

In [ ]:
print_report(y_train, y_train_pred_xg, thresh=0.5)

In [ ]:
print_report(y_valid, y_valid_pred_xg, thresh=0.5)

## Model 3: XGBoost GridSearch

In [ ]:
params = {
    'max_depth': [5, 7],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 150],
    'min_child_weight': [1, 2],
    'subsample': [0.7, 0.9]
}

In [ ]:
xgb = XGBClassifier(
    validate_parameters = True,
    objective='binary:logistic',
    nthread=-1)

In [ ]:
search = GridSearchCV(
    estimator=xgb,
    param_grid=params,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=2)

In [ ]:
search.fit(X_train_tf, y_train)

In [ ]:
search.best_params_

In [ ]:
best_model = search.best_estimator_
best_model

In [ ]:
y_treino_pred_xg_otim = best_model.predict(X_train_tf)
y_valid_pred_xg_otim = best_model.predict(X_valid_tf)
y_teste_pred_xg_otim = best_model.predict(X_test_tf)

In [ ]:
print_report(y_valid, y_valid_pred_xg_otim, 0.5)

In [ ]:
print_report(y_test, y_teste_pred_xg_otim, 0.5)

In [ ]:
fpr_train, tpr_train, thresholds_train = roc_curve(y_train, y_treino_pred_xg_otim)
auc_train = roc_auc_score(y_train, y_treino_pred_xg_otim)

In [ ]:
fpr_valid, tpr_valid, thresholds_valid = roc_curve(y_valid, y_valid_pred_xg_otim)
auc_valid = roc_auc_score(y_valid, y_valid_pred_xg_otim)

In [ ]:
fpr_test, tpr_test, thresholds_test = roc_curve(y_test, y_teste_pred_xg_otim)
auc_test = roc_auc_score(y_test, y_teste_pred_xg_otim)

In [ ]:
# Plot
plt.figure(figsize=(9,5))
plt.plot(fpr_train, tpr_train, 'r-', label = 'Train AUC %.3f' % auc_train)
plt.plot(fpr_valid, tpr_valid, 'b-', label = 'Validation AUC: %.3f' % auc_valid)
plt.plot(fpr_test, tpr_test, 'g-', label = 'Test AUC: %.3f' % auc_test)
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

# Inference

In [ ]:
new_df = pd.read_csv(f"{data_path}/new_data.csv")

In [ ]:
x_new = new_df[cols]
x_new

In [ ]:
x_new_tf = scaler.transform(x_new)

In [ ]:
if best_model.predict(x_new_tf).item() == 0:
    print(f"The machine does NOT need maintenance.")
else:
    print(f"The machine needs maintenance.")